# Task 1：实现 LIF 神经元

目标：模拟**一个神经元**接收恒定电流后，膜电位如何变化、何时发放脉冲。依次运行下方单元格即可。

## 1. 神经元在每一步做什么？

膜电位 $V$ 接收电流后上升，同时向静息电位 $V_{rest}$ 泄漏。按题目的 Euler 方法，步长 $\Delta t$ 内的变化是：

$$V_{t+\Delta t}=V_t+\frac{\Delta t}{\tau_m}[-(V_t-V_{rest})+RI_t]$$

达到阈值就记一个脉冲，并重置电位；接下来的不应期保持重置电位。下面使用 mV、nA、MΩ、ms；其中 $1\,\mathrm{M\Omega}\times 1\,\mathrm{nA}=1\,\mathrm{mV}$。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import snntorch as snn
import torch

## 2. 定义并运行一个 LIF 神经元

`snn.Leaky` 用相对静息电位的数值，因此将静息电位视为 0。`beta = 1 - dt/tau` 是每步保留的电位比例，`drive = dt/tau × R × I` 是输入电流带来的增量。发放之后，额外用计数器实现 2 ms 不应期。

snnTorch 默认以严格大于阈值发放；题目要求达到阈值即发放，所以这里用 `at_threshold` 指定 $\geq$。它只用于本次前向仿真，不用于训练。

In [ ]:
def at_threshold(voltage):
    return (voltage >= 0).to(voltage.dtype)


def simulate(current_na, duration_ms=2000, dt=0.1):
    rest, reset, threshold = -65.0, -65.0, -50.0  # mV
    tau, resistance, refractory_ms = 20.0, 10.0, 2.0  # ms、MΩ、ms
    neuron = snn.Leaky(
        beta=torch.tensor(1 - dt / tau, dtype=torch.float64),
        threshold=threshold - rest,
        reset_mechanism="zero",
        reset_delay=False,
        spike_grad=at_threshold,
    ).double()
    voltage = torch.tensor(reset - rest, dtype=torch.float64)
    drive = torch.tensor(dt / tau * resistance * current_na, dtype=torch.float64)
    remaining = 0
    voltages, spikes = [], []
    with torch.no_grad():
        for _ in range(round(duration_ms / dt)):
            if remaining > 0:  # 不应期：保持重置电位，不发放
                voltage = torch.tensor(reset - rest, dtype=torch.float64)
                spike = False
                remaining -= 1
            else:
                spike, voltage = neuron(drive, voltage)
                spike = bool(spike.item())
                if spike:
                    voltage = torch.tensor(reset - rest, dtype=torch.float64)
                    remaining = round(refractory_ms / dt)
            voltages.append(voltage.item() + rest)
            spikes.append(spike)
    return np.asarray(voltages), np.asarray(spikes)

## 3. 观察一次仿真

给神经元施加 2 nA 恒定电流，运行 2 秒。图中只展示前 200 ms，红虚线是发放阈值，圆点是脉冲发生的时刻。这里记录的是**每一步结束后的电位**，因此脉冲对应的电位已经重置。

In [ ]:
dt = 0.1
voltage, spikes = simulate(current_na=2.0, dt=dt)
time_ms = (np.arange(len(voltage)) + 1) * dt

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(time_ms, voltage, label="Membrane potential")
ax.axhline(-50, color="red", linestyle="--", label="Threshold")
ax.scatter(time_ms[spikes], voltage[spikes], s=8, label="Spike / reset")
ax.set(xlabel="Time (ms)", ylabel="Voltage (mV)")
ax.set_xlim(0, 200)
ax.legend()
plt.show()
print(
    f"脉冲数：{spikes.sum()}；平均发放率：{spikes.sum() / (len(spikes) * dt / 1000):.1f} Hz"
)

## 4. 用理论检查实现

默认参数下，临界恒定电流为 $(V_{th}-V_{rest})/R=1.5$ nA。低于临界值时电位达不到阈值；恰好等于临界值时电位渐近接近阈值，在有限时间内也不发放。将上面的 2 nA 改为 1 nA、1.5 nA 再运行，观察脉冲是否消失。

这里完成的是 Task 1。不同输入、系统对照实验和滑动发放率属于后续任务。